In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))

import torch
import json
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast

from notebooks.local.utils import get_paths, create_folders, load_progress, save_progress, mark_done, is_done

PATHS    = get_paths()
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = torch.cuda.is_available()
THRESHOLDS = [0.05, 0.10, 0.20]

TEMPERATURE = 15.0
LR          = 1e-4
NUM_EPOCHS  = 10
LORA_RANKS  = [2, 4, 8, 16]

create_folders(PATHS)
print("Device:", DEVICE, "  fp16:", USE_FP16)


In [ ]:
import shutil, tarfile
from notebooks.local.utils import get_paths, create_folders, download_file

PATHS    = get_paths()
create_folders(PATHS)

spair_check = os.path.join(PATHS['spair71k'], 'JPEGImages')
if not os.path.exists(spair_check):
    print('SPair-71k not found. Downloading (~2 GB) ...')
    tar_path = os.path.join(PATHS['data'], 'SPair-71k.tar.gz')
    download_file(
        'http://cvlab.postech.ac.kr/research/SPair-71k/data/SPair-71k.tar.gz',
        tar_path, desc='SPair-71k',
    )
    print('Extracting ...')
    with tarfile.open(tar_path, 'r:gz') as t:
        t.extractall(PATHS['spair71k'])
    extracted_sub = os.path.join(PATHS['spair71k'], 'SPair-71k')
    if os.path.isdir(extracted_sub):
        for item in os.listdir(extracted_sub):
            shutil.move(os.path.join(extracted_sub, item),
                        os.path.join(PATHS['spair71k'], item))
        os.rmdir(extracted_sub)
    os.remove(tar_path)
    print('SPair-71k ready.')
else:
    print('SPair-71k already present.')

if not os.path.exists(PATHS['dinov2_w']):
    print('Downloading DINOv2 ViT-B/14 weights (~330 MB) ...')
    download_file(
        'https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth',
        PATHS['dinov2_w'], desc='DINOv2',
    )
else:
    print('DINOv2 weights present.')

if not os.path.exists(PATHS['sam_w']):
    print('Downloading SAM ViT-B weights (~370 MB) ...')
    download_file(
        'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
        PATHS['sam_w'], desc='SAM',
    )
else:
    print('SAM weights present.')

if not os.path.exists(PATHS['dinov3_w']):
    print('WARNING: DINOv3 weights not found at', PATHS['dinov3_w'])
    print('  Place dinov3_vitb16_pretrain.pth in weights/ (obtain from project maintainer).')
else:
    print('DINOv3 weights present.')


In [ ]:
from src.models.dinov2.dinov2.models.vision_transformer import vit_base as vit_base_v2
from src.models.dinov3.dinov3.models.vision_transformer import vit_base as vit_base_v3
from src.datasets.spair_dataset import SPairDataset
from src.features.extractor import extract_dense_features, pixel_to_patch_coord, patch_to_pixel_coord
from src.lora.lora import inject_lora, remove_lora, count_trainable_params
from experiments.finetune import compute_cross_entropy_loss, validate
from experiments.evaluate import evaluate, save_results
import torch.nn.functional as F


def load_fresh(backbone, paths, device, use_fp16=False):
    if backbone == 'dinov2':
        model = vit_base_v2(img_size=(518,518), patch_size=14,
                            num_register_tokens=0, block_chunks=0, init_values=1.0)
        ckpt = torch.load(paths['dinov2_w'], map_location=device, weights_only=True)
        model.load_state_dict(ckpt, strict=True)
        img_size, patch_size = 518, 14
    elif backbone == 'dinov3':
        model = vit_base_v3(img_size=512, patch_size=16)
        ckpt = torch.load(paths['dinov3_w'], map_location=device, weights_only=True)
        model.load_state_dict(ckpt, strict=True)
        img_size, patch_size = 512, 16
    model = model.to(device)
    if use_fp16:
        model = model.half()
    model.eval()
    return model, img_size, patch_size


def train_epoch_fp16_lora(model, dataloader, optimizer, scaler, device,
                           img_size, patch_size, temperature, scheduler=None):
    model.train()
    total_loss, n = 0.0, 0
    for idx, sample in enumerate(dataloader):
        src_t = sample['src_img'].to(device)
        tgt_t = sample['trg_img'].to(device)
        src_t = F.interpolate(src_t, size=(img_size,img_size), mode='bilinear', align_corners=False)
        tgt_t = F.interpolate(tgt_t, size=(img_size,img_size), mode='bilinear', align_corners=False)
        if USE_FP16:
            src_t, tgt_t = src_t.half(), tgt_t.half()

        src_orig = (sample['src_imsize'][2], sample['src_imsize'][1])
        tgt_orig = (sample['trg_imsize'][2], sample['trg_imsize'][1])
        src_kps = sample['src_kps'].numpy()[0]
        trg_kps = sample['trg_kps'].numpy()[0]

        with autocast(enabled=USE_FP16):
            src_feat = extract_dense_features(model, src_t, training=True)
            tgt_feat = extract_dense_features(model, tgt_t, training=True)
            loss = compute_cross_entropy_loss(
                src_feat.float(), tgt_feat.float(),
                src_kps, trg_kps, src_orig, tgt_orig,
                img_size, patch_size, temperature,
            )

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if scheduler:
            scheduler.step()
        total_loss += loss.item(); n += 1
        if (idx + 1) % 100 == 0:
            print(f"  batch {idx+1}/{len(dataloader)}  loss={loss.item():.4f}")
    return total_loss / max(n, 1)


pair_ann = os.path.join(PATHS['spair71k'], 'PairAnnotation')
layout   = os.path.join(PATHS['spair71k'], 'Layout')
images   = os.path.join(PATHS['spair71k'], 'JPEGImages')
train_ds = SPairDataset(pair_ann, layout, images, 'large', 0.1, 'trn')
val_ds   = SPairDataset(pair_ann, layout, images, 'large', 0.1, 'val')
test_ds  = SPairDataset(pair_ann, layout, images, 'large', 0.1, 'test')
print(f"Datasets loaded — train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")


## LoRA Rank Ablation

In [ ]:
PROGRESS_PATH = os.path.join(PATHS['step4_lora'], 'progress_rank.json')
progress = load_progress(PROGRESS_PATH)

for backbone in ['dinov2', 'dinov3']:
    for r in LORA_RANKS:
        key = f"r{r}"
        if is_done(progress, backbone, 'rank_ablation', key):
            print(f"Skip {backbone} r={r}")
            continue

        out_dir = os.path.join(PATHS['step4_lora'], f"{backbone}_r{r}")
        os.makedirs(out_dir, exist_ok=True)

        model, img_size, patch_size = load_fresh(backbone, PATHS, DEVICE, USE_FP16)
        inject_lora(model, r=r, alpha=float(r))
        params = count_trainable_params(model)
        print(f"{backbone} r={r}: {params['trainable']:,} trainable / {params['total']:,} total ({params['percentage']:.2f}%)")

        optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
        loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=0)
        scaler = GradScaler(enabled=USE_FP16)

        best_pck, patience_count = 0.0, 0
        for epoch in range(1, NUM_EPOCHS + 1):
            train_epoch_fp16_lora(model, loader, optimizer, scaler, DEVICE,
                                  img_size, patch_size, TEMPERATURE)
            val_pck = validate(model, val_ds, DEVICE, img_size, patch_size)
            print(f"  epoch {epoch}: val_pck@0.10={val_pck:.2f}%")
            if val_pck > best_pck:
                best_pck = val_pck
                patience_count = 0
                torch.save({'model_state_dict': model.state_dict(), 'val_pck': val_pck, 'r': r},
                           os.path.join(out_dir, 'best.pth'))
            else:
                patience_count += 1
                if patience_count >= 2:
                    print(f"  Early stopping")
                    break

        result = {'backbone': backbone, 'r': r, 'best_val_pck': best_pck, 'trainable': params['trainable']}
        with open(os.path.join(out_dir, 'result.json'), 'w') as f:
            json.dump(result, f, indent=2)
        mark_done(progress, backbone, 'rank_ablation', key, PROGRESS_PATH)
        del model; torch.cuda.empty_cache()

print("LoRA rank ablation done.")


## Full LoRA Training (Best Rank)

In [ ]:
for backbone in ['dinov2', 'dinov3']:
    best_r = 4  # default; change based on ablation results above
    lora_ckpt = PATHS[f'{backbone}_lora']
    os.makedirs(os.path.dirname(lora_ckpt), exist_ok=True)

    if os.path.exists(lora_ckpt):
        print(f"{backbone} LoRA checkpoint exists: {lora_ckpt}")
        continue

    model, img_size, patch_size = load_fresh(backbone, PATHS, DEVICE, USE_FP16)
    inject_lora(model, r=best_r, alpha=float(best_r))

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
    loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=0)
    scaler = GradScaler(enabled=USE_FP16)
    best_pck, patience_count = 0.0, 0

    for epoch in range(1, NUM_EPOCHS + 1):
        train_epoch_fp16_lora(model, loader, optimizer, scaler, DEVICE,
                              img_size, patch_size, TEMPERATURE)
        val_pck = validate(model, val_ds, DEVICE, img_size, patch_size)
        print(f"Epoch {epoch}: val_pck@0.10={val_pck:.2f}%")
        if val_pck > best_pck:
            best_pck = val_pck
            patience_count = 0
            torch.save({'model_state_dict': model.state_dict(), 'val_pck': val_pck}, lora_ckpt)
        else:
            patience_count += 1
            if patience_count >= 2:
                print("Early stopping")
                break

    print(f"{backbone} LoRA best val PCK@0.10: {best_pck:.2f}%")
    del model; torch.cuda.empty_cache()


## LoRA Evaluation on SPair-71k Test

In [ ]:
lora_results = {}
for backbone in ['dinov2', 'dinov3']:
    lora_ckpt = PATHS[f'{backbone}_lora']
    out_dir = os.path.join(PATHS['step4_lora'], f"{backbone}_eval")
    os.makedirs(out_dir, exist_ok=True)
    stats_path = os.path.join(out_dir, 'overall_stats.json')

    if os.path.exists(stats_path):
        print(f"{backbone} LoRA eval — already done.")
        with open(stats_path) as f:
            lora_results[backbone] = json.load(f)
        continue

    if not os.path.exists(lora_ckpt):
        print(f"No LoRA checkpoint for {backbone}, skipping.")
        continue

    model, img_size, patch_size = load_fresh(backbone, PATHS, DEVICE, USE_FP16)
    ckpt = torch.load(lora_ckpt, map_location=DEVICE, weights_only=True)
    state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    model.load_state_dict(state, strict=False)  # LoRA params are in state_dict
    model.eval()

    per_img, all_kp, t = evaluate(model, test_ds, DEVICE, THRESHOLDS)
    save_results(per_img, all_kp, out_dir, t, THRESHOLDS)
    with open(stats_path) as f:
        lora_results[backbone] = json.load(f)
    del model; torch.cuda.empty_cache()
    print(f"{backbone} LoRA eval done.")


## Summary: Baseline vs Fine-tuned vs LoRA

In [ ]:
rows = []
for backbone in ['dinov2', 'dinov3']:
    for variant, path_key in [
        ('baseline', os.path.join(PATHS['step1'], f'{backbone}_argmax')),
        ('finetuned', os.path.join(PATHS['step2'], f'{backbone}_finetuned')),
        ('LoRA',      os.path.join(PATHS['step4_lora'], f'{backbone}_eval')),
    ]:
        stats_path = os.path.join(path_key, 'overall_stats.json')
        if os.path.exists(stats_path):
            with open(stats_path) as f:
                s = json.load(f)
            rows.append({
                'Model': f"{backbone} {variant}",
                'PCK@0.05': round(s.get('pck@0.05',{}).get('mean', float('nan')), 2),
                'PCK@0.10': round(s.get('pck@0.10',{}).get('mean', float('nan')), 2),
                'PCK@0.20': round(s.get('pck@0.20',{}).get('mean', float('nan')), 2),
            })
        else:
            rows.append({'Model': f"{backbone} {variant}", 'PCK@0.05': '-', 'PCK@0.10': '-', 'PCK@0.20': '-'})

df = pd.DataFrame(rows).set_index('Model')
print(df.to_string())
